Imagecodec is needed, or can use the requirements.txt to install all packages (not recommended)

In [ ]:
!pip install imagecodecs

Upload Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/hw3-data-release.tar /content/
!tar -xf hw3-data-release.tar

Training


In [ ]:
import os
import random
import numpy as np
import tifffile
import torch
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import MaskRCNN
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models import ResNet101_Weights
from torchvision.models import resnext101_32x8d, ResNeXt101_32X8D_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator
from skimage import measure
from tqdm.notebook import tqdm
from torchvision.models.detection.rpn import RPNHead

# DATASET

class CellDataset(Dataset):
    def __init__(self, root_dir, tile_size=256, multiplier=8, augment=True):
        self.root_dir = root_dir
        self.tile_size = tile_size
        self.multiplier = multiplier
        self.augment = augment
        self.subfolders = [
            os.path.join(root_dir, d) for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ]

    def __len__(self):
        return len(self.subfolders) * self.multiplier

    def __getitem__(self, idx):
        actual_idx = idx % len(self.subfolders)
        folder = self.subfolders[actual_idx]

        try:
            img_raw = tifffile.imread(os.path.join(folder, "image.tif")).astype(np.float32)

            if img_raw.ndim == 2:
                img_raw = np.stack([img_raw] * 3, axis=-1)
            elif img_raw.ndim == 3 and img_raw.shape[-1] == 4:
                img_raw = img_raw[:, :, :3]

            h, w = img_raw.shape[:2]

            # Padding
            if h < self.tile_size or w < self.tile_size:
                pad_h = max(0, self.tile_size - h)
                pad_w = max(0, self.tile_size - w)
                img_raw = np.pad(img_raw, ((0, pad_h), (0, pad_w), (0, 0)), mode='constant')
                h, w = img_raw.shape[:2]

            # Crop & Augmentation
            y1 = random.randint(0, h - self.tile_size)
            x1 = random.randint(0, w - self.tile_size)
            y2, x2 = y1 + self.tile_size, x1 + self.tile_size

            do_hflip = self.augment and (random.random() > 0.5)
            do_vflip = self.augment and (random.random() > 0.5)
            rot_k = random.choice([0, 1, 2, 3]) if self.augment else 0

            img_tile = img_raw[y1:y2, x1:x2]
            if do_hflip: img_tile = np.fliplr(img_tile)
            if do_vflip: img_tile = np.flip_ud(img_tile)
            if rot_k > 0: img_tile = np.rot90(img_tile, k=rot_k, axes=(0, 1))

            img_max = img_tile.max()
            denom = img_max if img_max > 1.0 else 1.0
            img_tensor = torch.from_numpy(img_tile.copy()).permute(2, 0, 1) / denom

            if self.augment:
                img_tensor = TF.adjust_brightness(img_tensor, random.uniform(0.9, 1.1))
                img_tensor = TF.adjust_contrast(img_tensor, random.uniform(0.9, 1.1))

            masks, labels = [], []
            for i in range(1, 5):
                m_path = os.path.join(folder, f"class{i}.tif")
                if not os.path.exists(m_path): continue

                m_raw = tifffile.imread(m_path)

                if m_raw.shape[0] < h or m_raw.shape[1] < w:
                    m_raw = np.pad(m_raw, ((0, max(0, h-m_raw.shape[0])), (0, max(0, w-m_raw.shape[1]))), mode='constant')

                m_tile = m_raw[y1:y2, x1:x2]
                if do_hflip: m_tile = np.fliplr(m_tile)
                if do_vflip: m_tile = np.flip_ud(m_tile)
                if rot_k > 0: m_tile = np.rot90(m_tile, k=rot_k, axes=(0, 1))

                if m_tile.max() == 0: continue

                labeled = measure.label(m_tile > 0)
                for inst_id in range(1, labeled.max() + 1):
                    inst_mask = (labeled == inst_id)
                    if inst_mask.sum() < 20: continue
                    masks.append(inst_mask)
                    labels.append(i)

            if len(masks) == 0:
                return self.__getitem__(random.randint(0, len(self.subfolders)-1))

            masks_tensor = torch.as_tensor(np.stack(masks), dtype=torch.uint8)
            labels_tensor = torch.as_tensor(labels, dtype=torch.int64)
            boxes = []
            for m in masks_tensor:
                pos = torch.where(m)
                boxes.append([torch.min(pos[1]), torch.min(pos[0]), torch.max(pos[1]), torch.max(pos[0])])

            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            keep = (boxes[:, 3] > boxes[:, 1]) & (boxes[:, 2] > boxes[:, 0])

            target = {
                "boxes": boxes[keep],
                "labels": labels_tensor[keep],
                "masks": masks_tensor[keep],
                "image_id": torch.tensor([actual_idx]),
                "area": (boxes[keep, 3] - boxes[keep, 1]) * (boxes[keep, 2] - boxes[keep, 0]),
                "iscrowd": torch.zeros((len(boxes[keep]),), dtype=torch.int64)
            }

            return img_tensor, target

        except Exception as e:
            return self.__getitem__(random.randint(0, len(self.subfolders)-1))

# MODEL

def get_model(num_classes):
    backbone = resnet_fpn_backbone(
        backbone_name='resnext101_32x8d',
        weights=ResNeXt101_32X8D_Weights.DEFAULT,
        trainable_layers=3
    )

    anchor_sizes = ((4,), (8,), (16,), (32,), (64,))
    aspect_ratios = ((0.2, 0.5, 1.0, 2.0, 5.0),) * len(anchor_sizes)
    rpn_anchor_generator = AnchorGenerator(anchor_sizes, aspect_ratios)

    model = MaskRCNN(
        backbone,
        num_classes=num_classes,
        rpn_anchor_generator=rpn_anchor_generator,
        box_detections_per_img=1500,
        box_score_thresh=0.01,
        box_nms_thresh=0.4,
        rpn_pre_nms_top_n_train=4000,
        rpn_post_nms_top_n_train=3000,
        rpn_pre_nms_top_n_test=4000,
        rpn_post_nms_top_n_test=3000,
        rpn_batch_size_per_image=512
    )

    out_channels = backbone.out_channels
    num_anchors = rpn_anchor_generator.num_anchors_per_location()[0]
    model.rpn.head = RPNHead(out_channels, num_anchors)

    in_f = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_f, num_classes)

    in_fm = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_fm, 512, num_classes)

    model.roi_heads.mask_roi_pool.output_size = (32, 32)
    model.transform.min_size = (256,)
    model.transform.max_size = 256

    return model

def collate_fn(batch): return tuple(zip(*batch))

# TRAINING

if __name__ == "__main__":
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    num_classes = 5

    model = get_model(num_classes)
    model.to(device)

    dataset = CellDataset('/content/train', tile_size=256, multiplier=8, augment=True)
    data_loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=4, collate_fn=collate_fn)

    total_epochs = 100
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs, eta_min=1e-7)

    best_loss = 10000
    for epoch in range(total_epochs):
        model.train()
        epoch_loss = 0
        pbar = tqdm(data_loader, desc=f"Epoch {epoch+1}/{total_epochs}")

        for images, targets in pbar:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            l_cls = loss_dict['loss_classifier']
            l_box = loss_dict['loss_box_reg'] * 1.5
            l_mask = loss_dict['loss_mask'] * 2.5
            l_obj = loss_dict['loss_objectness']
            l_rpn = loss_dict['loss_rpn_box_reg']

            losses = l_cls + l_box + l_mask + l_obj + l_rpn

            optimizer.zero_grad()
            losses.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += losses.item()
            pbar.set_postfix({
                "total": f"{losses.item():.3f}",
                "mask": f"{l_mask.item():.3f}",
                "box": f"{l_box.item():.3f}",
                "obj": f"{l_obj.item():.3f}"
            })

        scheduler.step()
        avg_loss = epoch_loss / len(data_loader)
        print(f"  > Epoch {epoch+1} Avg Loss: {avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), "model.pth")

Inference

In [ ]:
import os
import json
import torch
import tifffile
import numpy as np
from tqdm import tqdm
from pycocotools import mask as mask_util
import torchvision
from torchvision.ops import nms, batched_nms

# Configuration
TEST_IMG_DIR = '/content/test_release'
MAPPING_FILE = 'test_image_name_to_ids.json'
MODEL_PATH = 'model.pth'
OUTPUT_FILE = 'test-results.json'
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Tile Settings
TILE_SIZE = 256
STRIDE = 64
MARGIN = 20

@torch.no_grad()
def run_inference():
    with open(MAPPING_FILE, 'r') as f:
        image_mappings = json.load(f)

    num_classes = 5

    model = get_model(num_classes)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()

    all_results = []

    for img_info in tqdm(image_mappings, desc="Tiling Inference"):
        file_name = img_info['file_name']
        img_id = img_info['id']
        H, W = img_info['height'], img_info['width']

        path = os.path.join(TEST_IMG_DIR, file_name)
        if not os.path.exists(path):
            continue

        img_raw = tifffile.imread(path)
        if img_raw.ndim == 2:
            img_raw = np.stack([img_raw] * 3, axis=-1)
        elif img_raw.ndim == 3 and img_raw.shape[-1] == 4:
            img_raw = img_raw[:, :, :3]

        denom = 65535.0 if img_raw.dtype == np.uint16 else 255.0
        img_full_tensor = torch.from_numpy(img_raw.astype(np.float32) / denom).permute(2, 0, 1)

        temp_boxes = []
        temp_scores = []
        temp_labels = []
        temp_masks = []
        temp_origins = []

        # Sliding Window
        for y in range(0, H, STRIDE):
            for x in range(0, W, STRIDE):
                y1 = min(y, H - TILE_SIZE) if H > TILE_SIZE else 0
                x1 = min(x, W - TILE_SIZE) if W > TILE_SIZE else 0
                y2, x2 = y1 + TILE_SIZE, x1 + TILE_SIZE

                tile = img_full_tensor[:, y1:y2, x1:x2].to(DEVICE)
                prediction = model([tile])[0]

                boxes = prediction['boxes']
                scores = prediction['scores']

                is_inner = torch.ones(len(boxes), dtype=torch.bool, device=DEVICE)

                if x1 > 0: is_inner &= (boxes[:, 0] > MARGIN)
                if y1 > 0: is_inner &= (boxes[:, 1] > MARGIN)
                if x2 < W: is_inner &= (boxes[:, 2] < TILE_SIZE - MARGIN)
                if y2 < H: is_inner &= (boxes[:, 3] < TILE_SIZE - MARGIN)

                keep = (scores > 0.05) & is_inner
                if not keep.any():
                    continue

                s_boxes = boxes[keep].cpu()
                s_boxes[:, [0, 2]] += x1
                s_boxes[:, [1, 3]] += y1

                temp_boxes.append(s_boxes)
                temp_scores.append(scores[keep].cpu())
                temp_labels.append(prediction['labels'][keep].cpu())
                temp_masks.append(prediction['masks'][keep].cpu())
                temp_origins.append([(x1, y1)] * len(s_boxes))

        # Global Merge with NMS
        if len(temp_boxes) > 0:
            boxes = torch.cat(temp_boxes)
            scores = torch.cat(temp_scores)
            labels = torch.cat(temp_labels)

            keep_idx = batched_nms(boxes, scores, labels, iou_threshold=0.35)

            flat_masks = torch.cat(temp_masks)
            flat_origins = [item for sublist in temp_origins for item in sublist]

            for idx in keep_idx:
                score = float(scores[idx].item())
                label = int(labels[idx].item())
                bbox = boxes[idx].tolist()

                w_box = bbox[2] - bbox[0]
                h_box = bbox[3] - bbox[1]

                x_off, y_off = flat_origins[idx]
                local_mask = flat_masks[idx, 0].numpy()

                global_mask = np.zeros((H, W), dtype=np.uint8, order='F')
                m_h, m_w = local_mask.shape

                global_mask[y_off:y_off+m_h, x_off:x_off+m_w] = (local_mask > 0.35).astype(np.uint8)

                rle = mask_util.encode(global_mask)
                rle['counts'] = rle['counts'].decode('utf-8')

                all_results.append({
                    "image_id": int(img_id),
                    "bbox": [bbox[0], bbox[1], w_box, h_box],
                    "score": score,
                    "category_id": label,
                    "segmentation": rle
                })

    with open(OUTPUT_FILE, 'w') as f:
        json.dump(all_results, f)

    print(f"\nDone! Total detections: {len(all_results)}")

if __name__ == "__main__":
    run_inference()